# UrbanFloodBench — Approach 4: Physics-Residual Hybrid\n
\n
**Idea:** Predict a residual correction on top of a persistence-physics baseline.\n
\n
At each step: `delta_total = delta_physics_baseline + delta_residual_model`\n
\n
The baseline captures smooth diffusion/coupling/rain forcing; the network learns what baseline misses.

## Cell 1 — Install dependencies

In [ ]:
import subprocess, sys, torch

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

print(f'PyTorch: {torch.__version__}')
pip('torch_geometric', 'pyarrow', 'tqdm')
print('Done.')

## Cell 2 — Imports, config, GPU check

In [ ]:
import gc, json, random, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import amp
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch_geometric.nn import HGTConv
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'GPU is required. Enable GPU in Kaggle settings.'
print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'VRAM: {VRAM_GB:.1f} GB')

PREP_ROOT = Path('/kaggle/input/datasets/asadjabbar/urbanfloodbench-preprocessed-v1/preprocessed')
OUT_DIR   = Path('/kaggle/working')
OUT_DIR.mkdir(exist_ok=True)

CFG = {
    'lag_window':       6,
    'hidden_dim':       128,
    'heads':            4,
    'n_hgt_layers':     2,
    'dropout':          0.1,
    'epochs':           30,
    'lr':               3e-4,
    'lr_backbone':      1e-4,
    'lr_head':          5e-4,
    'grad_clip':        1.0,
    'warmup_steps':     10,
    'accum_steps':      4,
    'ss_decay_epochs':  20,
    'multistep_ks':     [1, 5, 10],
    'multistep_w':      [1.0, 0.5, 0.25],
    'n_val_events':     8,
    'freeze_backbone_epochs': 4,
    'use_amp':          True,
    'seed':             42
}

# Auto-switch to a lighter profile on small GPUs to prevent OOM in epoch 1.
if VRAM_GB < 20:
    CFG.update({
        'lag_window': 3,
        'hidden_dim': 64,
        'heads': 2,
        'n_hgt_layers': 1,
        'n_val_events': 3
    })
    print('Applied low-VRAM profile for stability.')

PHYS = {
    # Baseline in normalized-delta space
    'k_rain_1d': 0.06,
    'k_rain_2d': 0.10,
    'k_diff_1d': 0.22,
    'k_diff_2d': 0.18,
    'k_cpl_1d':  0.10,
    'k_cpl_2d':  0.10
}

torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])
random.seed(CFG['seed'])
torch.cuda.manual_seed_all(CFG['seed'])

print('Config:', json.dumps(CFG, indent=2))
print('Physics baseline coeffs:', json.dumps(PHYS, indent=2))

## Cell 3 — Data and graph helpers

In [ ]:
def load_model_config(model_id):
    out_dir = PREP_ROOT / f'model_{model_id}'
    with open(out_dir / 'feature_config.json') as f:
        fc = json.load(f)

    st = torch.load(out_dir / 'static_tensors.pt', map_location=DEVICE)
    static = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in st.items()}

    train_pts = sorted((out_dir / 'train').glob('event_*.pt'), key=lambda p: int(p.stem.split('_')[1]))
    test_pts  = sorted((out_dir / 'test').glob('event_*.pt'),  key=lambda p: int(p.stem.split('_')[1]))

    n_val = CFG['n_val_events']
    split = {'train': train_pts[:-n_val], 'val': train_pts[-n_val:], 'test': test_pts}

    print(f'Model {model_id}: train={len(split["train"])} val={len(split["val"])} test={len(split["test"])}')
    return fc, static, split

def load_event(path):
    # Keep events on CPU to avoid VRAM spikes from whole-event preloading.
    return torch.load(path, map_location='cpu')

def make_edge_dict(static):
    return {
        ('1d_node', 'pipe',       '1d_node'): static['edge_idx_1d'],
        ('2d_node', 'surface',    '2d_node'): static['edge_idx_2d'],
        ('1d_node', 'to_surface', '2d_node'): static['edge_idx_1d2d'],
        ('2d_node', 'to_drain',   '1d_node'): static['edge_idx_2d1d']
    }

DEPTH_IDX = 0
LAG_IDXS = [1, 2, 3]

def update_dyn(x_prev, new_depth_norm, x_at_t):
    x_new = x_at_t.clone()
    x_new[:, DEPTH_IDX] = new_depth_norm
    x_new[:, LAG_IDXS[0]] = x_prev[:, DEPTH_IDX]
    x_new[:, LAG_IDXS[1]] = x_prev[:, LAG_IDXS[0]]
    x_new[:, LAG_IDXS[2]] = x_prev[:, LAG_IDXS[1]]
    return x_new

def _make_windows(x_seq, t, lag):
    idxs = [max(0, t - lag + 1 + k) for k in range(lag)]
    return x_seq[idxs].permute(1, 0, 2).contiguous()

def _find_rain_index(dyn_cols):
    # Prefer current rainfall column over cumulative/derivative rain features
    for i, c in enumerate(dyn_cols):
        lc = c.lower()
        if 'rain' in lc and 'cum' not in lc and 'delta' not in lc:
            return i
    for i, c in enumerate(dyn_cols):
        if 'rain' in c.lower():
            return i
    return None

def _neighbor_mean(x, edge_idx, n_nodes):
    src, dst = edge_idx[0], edge_idx[1]
    out = torch.zeros(n_nodes, device=x.device, dtype=x.dtype)
    cnt = torch.zeros(n_nodes, device=x.device, dtype=x.dtype)
    out.index_add_(0, dst, x[src])
    cnt.index_add_(0, dst, torch.ones_like(x[src]))
    return out / cnt.clamp(min=1.0)

print('Helpers ready.')

## Cell 4 — Physics baseline + residual model

In [ ]:
def physics_baseline_delta_norm(prev_d1_norm, prev_d2_norm, x1_t, x2_t, static, rain_idx_1d, rain_idx_2d):
    n1 = prev_d1_norm.shape[0]
    n2 = prev_d2_norm.shape[0]

    prev_d1_norm = torch.nan_to_num(prev_d1_norm, nan=0.0, posinf=0.0, neginf=0.0)
    prev_d2_norm = torch.nan_to_num(prev_d2_norm, nan=0.0, posinf=0.0, neginf=0.0)

    mean_nbr_1d = _neighbor_mean(prev_d1_norm, static['edge_idx_1d'], n1)
    mean_nbr_2d = _neighbor_mean(prev_d2_norm, static['edge_idx_2d'], n2)

    mean_2d_to_1d = _neighbor_mean(prev_d2_norm, static['edge_idx_2d1d'], n1)
    mean_1d_to_2d = _neighbor_mean(prev_d1_norm, static['edge_idx_1d2d'], n2)

    rain1 = torch.zeros_like(prev_d1_norm) if rain_idx_1d is None else torch.nan_to_num(x1_t[:, rain_idx_1d], nan=0.0, posinf=0.0, neginf=0.0)
    rain2 = torch.zeros_like(prev_d2_norm) if rain_idx_2d is None else torch.nan_to_num(x2_t[:, rain_idx_2d], nan=0.0, posinf=0.0, neginf=0.0)

    diff1 = mean_nbr_1d - prev_d1_norm
    diff2 = mean_nbr_2d - prev_d2_norm
    cpl1 = mean_2d_to_1d - prev_d1_norm
    cpl2 = mean_1d_to_2d - prev_d2_norm

    base_d1 = PHYS['k_rain_1d'] * rain1 + PHYS['k_diff_1d'] * diff1 + PHYS['k_cpl_1d'] * cpl1
    base_d2 = PHYS['k_rain_2d'] * rain2 + PHYS['k_diff_2d'] * diff2 + PHYS['k_cpl_2d'] * cpl2

    # Keep baseline numerically stable in normalized-delta space
    base_d1 = torch.nan_to_num(base_d1, nan=0.0, posinf=0.0, neginf=0.0).clamp(min=-5.0, max=5.0)
    base_d2 = torch.nan_to_num(base_d2, nan=0.0, posinf=0.0, neginf=0.0).clamp(min=-5.0, max=5.0)
    return base_d1, base_d2

class LagTemporalEncoder(nn.Module):
    def __init__(self, f_dyn, hidden_dim, lag_window, dropout=0.1):
        super().__init__()
        self.lag_window = lag_window
        self.in_proj = nn.Linear(f_dyn, hidden_dim)
        self.pos = nn.Parameter(torch.randn(lag_window, hidden_dim) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=4, dim_feedforward=hidden_dim * 4,
            dropout=dropout, batch_first=True, activation='gelu')
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=1)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x_win):
        h = self.in_proj(x_win)
        h = h + self.pos.unsqueeze(0)
        h = self.encoder(h)
        return self.norm(h[:, -1, :])

class ResidualHybridHGT(nn.Module):
    def __init__(self, f_1d_dyn, f_2d_dyn, f_1d_static, f_2d_static, lag_window, hidden_dim, heads, n_hgt_layers, dropout):
        super().__init__()
        self.temporal_1d = LagTemporalEncoder(f_1d_dyn, hidden_dim, lag_window, dropout)
        self.temporal_2d = LagTemporalEncoder(f_2d_dyn, hidden_dim, lag_window, dropout)

        self.static_1d_proj = nn.Linear(f_1d_static + 3, hidden_dim)
        self.static_2d_proj = nn.Linear(f_2d_static + 3, hidden_dim)

        self.merge_1d = nn.Sequential(nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(), nn.Dropout(dropout))
        self.merge_2d = nn.Sequential(nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(), nn.Dropout(dropout))

        self.hgt_layers = nn.ModuleList([
            HGTConv(
                in_channels=hidden_dim,
                out_channels=hidden_dim,
                metadata=(["1d_node", "2d_node"],
                          [("1d_node", "pipe", "1d_node"),
                           ("2d_node", "surface", "2d_node"),
                           ("1d_node", "to_surface", "2d_node"),
                           ("2d_node", "to_drain", "1d_node")]),
                heads=heads
            )
            for _ in range(n_hgt_layers)
        ])
        self.norm_1d = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(n_hgt_layers)])
        self.norm_2d = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(n_hgt_layers)])
        self.drop = nn.Dropout(dropout)

        # Residual head predicts correction in normalized-delta space
        self.head_1d = nn.Sequential(nn.Linear(hidden_dim + 1, hidden_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim, 1))
        self.head_2d = nn.Sequential(nn.Linear(hidden_dim + 1, hidden_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim, 1))

    def forward_step(self, x1_win, x2_win, static_1d, static_2d, time_t, edge_dict, base_delta_1d, base_delta_2d):
        t1 = time_t.unsqueeze(0).expand(static_1d.shape[0], -1)
        t2 = time_t.unsqueeze(0).expand(static_2d.shape[0], -1)

        h1_temp = self.temporal_1d(x1_win)
        h2_temp = self.temporal_2d(x2_win)
        h1_stat = self.static_1d_proj(torch.cat([static_1d, t1], dim=-1))
        h2_stat = self.static_2d_proj(torch.cat([static_2d, t2], dim=-1))

        h1 = self.merge_1d(torch.cat([h1_temp, h1_stat], dim=-1))
        h2 = self.merge_2d(torch.cat([h2_temp, h2_stat], dim=-1))

        x_dict = {'1d_node': h1, '2d_node': h2}
        for i, hgt in enumerate(self.hgt_layers):
            out = hgt(x_dict, edge_dict)
            x_dict['1d_node'] = self.norm_1d[i](x_dict['1d_node'] + self.drop(out['1d_node']))
            x_dict['2d_node'] = self.norm_2d[i](x_dict['2d_node'] + self.drop(out['2d_node']))

        inp1 = torch.cat([x_dict['1d_node'], base_delta_1d.unsqueeze(-1)], dim=-1)
        inp2 = torch.cat([x_dict['2d_node'], base_delta_2d.unsqueeze(-1)], dim=-1)
        res1 = self.head_1d(inp1).squeeze(-1)
        res2 = self.head_2d(inp2).squeeze(-1)

        # Total normalized delta = baseline + learned correction
        d1_total = base_delta_1d + res1
        d2_total = base_delta_2d + res2
        d1_total = torch.nan_to_num(d1_total, nan=0.0, posinf=0.0, neginf=0.0).clamp(min=-8.0, max=8.0)
        d2_total = torch.nan_to_num(d2_total, nan=0.0, posinf=0.0, neginf=0.0).clamp(min=-8.0, max=8.0)
        return d1_total, d2_total, res1, res2

def build_model(fc):
    m = ResidualHybridHGT(
        f_1d_dyn=fc['f_1d_dyn'],
        f_2d_dyn=fc['f_2d_dyn'],
        f_1d_static=fc['f_1d_static'],
        f_2d_static=fc['f_2d_static'],
        lag_window=CFG['lag_window'],
        hidden_dim=CFG['hidden_dim'],
        heads=CFG['heads'],
        n_hgt_layers=CFG['n_hgt_layers'],
        dropout=CFG['dropout']
    ).to(DEVICE)
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'Model parameters: {n:,}')
    return m

print('Physics residual model ready.')

## Cell 5 — Rollout, validation, training

In [ ]:
def rollout_event(model, ev, static, edge_dict, fc, rain_idx_1d, rain_idx_2d, p_teacher=1.0, train=True, multistep_ks=None, multistep_w=None):
    if multistep_ks is None:
        multistep_ks, multistep_w = [1], [1.0]

    WARMUP = CFG['warmup_steps']
    L = CFG['lag_window']

    x1_all = torch.nan_to_num(ev['x_1d'].to(DEVICE, non_blocking=True), nan=0.0, posinf=0.0, neginf=0.0)
    x2_all = torch.nan_to_num(ev['x_2d'].to(DEVICE, non_blocking=True), nan=0.0, posinf=0.0, neginf=0.0)
    y1 = torch.nan_to_num(ev['delta_1d'].to(DEVICE, non_blocking=True), nan=0.0, posinf=0.0, neginf=0.0)
    y2 = torch.nan_to_num(ev['delta_2d'].to(DEVICE, non_blocking=True), nan=0.0, posinf=0.0, neginf=0.0)
    depth1 = torch.nan_to_num(ev['depth_1d'].to(DEVICE, non_blocking=True), nan=0.0, posinf=0.0, neginf=0.0).clamp(min=0.0)
    depth2 = torch.nan_to_num(ev['depth_2d'].to(DEVICE, non_blocking=True), nan=0.0, posinf=0.0, neginf=0.0).clamp(min=0.0)
    time_enc = torch.nan_to_num(ev['time_enc'].to(DEVICE, non_blocking=True), nan=0.0, posinf=0.0, neginf=0.0)
    T = ev['T']

    d1_mean, d1_std = fc['depth_mean_1d'], fc['std_depth_1d'] + 1e-8
    d2_mean, d2_std = fc['depth_mean_2d'], fc['std_depth_2d'] + 1e-8
    dd1_std, dd2_std = fc['std_delta_1d'] + 1e-8, fc['std_delta_2d'] + 1e-8

    cur_x1 = x1_all.clone()
    cur_x2 = x2_all.clone()
    pred_d1 = depth1[WARMUP - 1].clone()
    pred_d2 = depth2[WARMUP - 1].clone()

    total_loss = torch.tensor(0.0, device=DEVICE)
    loss_count = 0

    for t in range(WARMUP, T):
        prev_d1_norm = cur_x1[t - 1, :, DEPTH_IDX]
        prev_d2_norm = cur_x2[t - 1, :, DEPTH_IDX]
        base1, base2 = physics_baseline_delta_norm(prev_d1_norm, prev_d2_norm, cur_x1[t], cur_x2[t], static, rain_idx_1d, rain_idx_2d)

        x1_win = _make_windows(cur_x1, t - 1, L)
        x2_win = _make_windows(cur_x2, t - 1, L)
        d1_tot, d2_tot, _, _ = model.forward_step(
            x1_win, x2_win,
            static['static_1d_node'], static['static_2d_node'],
            time_enc[t], edge_dict, base1, base2
        )

        if train:
            step_loss = 0.0
            cnt = 0
            for k, w in zip(multistep_ks, multistep_w):
                tk = t + k - 1
                if tk < T:
                    l = 0.5 * F.mse_loss(d1_tot, y1[tk]) + 0.5 * F.mse_loss(d2_tot, y2[tk])
                    if torch.isfinite(l):
                        step_loss = step_loss + w * l
                        cnt += 1
            if cnt > 0:
                total_loss = total_loss + step_loss / cnt
                loss_count += 1

        new_d1 = (pred_d1 + d1_tot.detach() * dd1_std).clamp(min=0.0)
        new_d2 = (pred_d2 + d2_tot.detach() * dd2_std).clamp(min=0.0)

        if random.random() < p_teacher:
            src_d1, src_d2 = depth1[t], depth2[t]
        else:
            src_d1, src_d2 = new_d1, new_d2

        src_d1_norm = (src_d1 - d1_mean) / d1_std
        src_d2_norm = (src_d2 - d2_mean) / d2_std

        t_next = min(t + 1, T - 1)
        cur_x1[t_next] = update_dyn(cur_x1[t], src_d1_norm, x1_all[t_next])
        cur_x2[t_next] = update_dyn(cur_x2[t], src_d2_norm, x2_all[t_next])
        pred_d1, pred_d2 = new_d1, new_d2

    if loss_count > 0:
        total_loss = total_loss / loss_count
    return total_loss

def standardised_rmse(pred, gt, std):
    return float(np.sqrt(np.mean((pred - gt) ** 2)) / (std + 1e-8))

@torch.no_grad()
def validate(model, val_paths, static, edge_dict, fc, rain_idx_1d, rain_idx_2d):
    model.eval()
    WARMUP = CFG['warmup_steps']
    L = CFG['lag_window']

    d1_mean, d1_std = fc['depth_mean_1d'], fc['std_depth_1d'] + 1e-8
    d2_mean, d2_std = fc['depth_mean_2d'], fc['std_depth_2d'] + 1e-8
    dd1_std, dd2_std = fc['std_delta_1d'] + 1e-8, fc['std_delta_2d'] + 1e-8

    invert = static['invert_elevation'].cpu().numpy()
    minelev = static['min_elevation'].cpu().numpy()

    s1, s2 = [], []
    n_used = 0
    n_skipped = 0
    for p in val_paths:
        ev = load_event(p)
        T = ev['T']
        x1_all = torch.nan_to_num(ev['x_1d'].to(DEVICE, non_blocking=True), nan=0.0, posinf=0.0, neginf=0.0)
        x2_all = torch.nan_to_num(ev['x_2d'].to(DEVICE, non_blocking=True), nan=0.0, posinf=0.0, neginf=0.0)
        x1 = x1_all.clone()
        x2 = x2_all.clone()
        depth1 = torch.nan_to_num(ev['depth_1d'].to(DEVICE, non_blocking=True), nan=0.0, posinf=0.0, neginf=0.0).clamp(min=0.0)
        depth2 = torch.nan_to_num(ev['depth_2d'].to(DEVICE, non_blocking=True), nan=0.0, posinf=0.0, neginf=0.0).clamp(min=0.0)
        time_enc = torch.nan_to_num(ev['time_enc'].to(DEVICE, non_blocking=True), nan=0.0, posinf=0.0, neginf=0.0)

        pred_d1 = depth1[WARMUP - 1].clone()
        pred_d2 = depth2[WARMUP - 1].clone()
        preds1, preds2 = [], []
        bad = False

        for t in range(WARMUP, T):
            prev_d1_norm = x1[t - 1, :, DEPTH_IDX]
            prev_d2_norm = x2[t - 1, :, DEPTH_IDX]
            base1, base2 = physics_baseline_delta_norm(prev_d1_norm, prev_d2_norm, x1[t], x2[t], static, rain_idx_1d, rain_idx_2d)

            x1_win = _make_windows(x1, t - 1, L)
            x2_win = _make_windows(x2, t - 1, L)
            d1_tot, d2_tot, _, _ = model.forward_step(
                x1_win, x2_win,
                static['static_1d_node'], static['static_2d_node'],
                time_enc[t], edge_dict, base1, base2
            )

            nd1 = (pred_d1 + d1_tot * dd1_std).clamp(min=0.0)
            nd2 = (pred_d2 + d2_tot * dd2_std).clamp(min=0.0)
            if (not torch.isfinite(nd1).all()) or (not torch.isfinite(nd2).all()):
                bad = True
                break

            preds1.append(nd1.cpu().numpy())
            preds2.append(nd2.cpu().numpy())

            nd1_norm = (nd1 - d1_mean) / d1_std
            nd2_norm = (nd2 - d2_mean) / d2_std
            t_next = min(t + 1, T - 1)
            x1[t_next] = update_dyn(x1[t], nd1_norm, x1_all[t_next])
            x2[t_next] = update_dyn(x2[t], nd2_norm, x2_all[t_next])
            pred_d1, pred_d2 = nd1, nd2

        if bad or len(preds1) == 0:
            n_skipped += 1
            del ev
            gc.collect()
            continue

        pd1 = np.stack(preds1)
        pd2 = np.stack(preds2)
        gd1 = depth1[WARMUP:].cpu().numpy()
        gd2 = depth2[WARMUP:].cpu().numpy()

        pw1 = pd1 + invert[np.newaxis, :]
        pw2 = pd2 + minelev[np.newaxis, :]
        gw1 = gd1 + invert[np.newaxis, :]
        gw2 = gd2 + minelev[np.newaxis, :]

        r1 = np.mean([standardised_rmse(pw1[:, n], gw1[:, n], fc['std_wl_1d']) for n in range(fc['n_1d'])])
        r2 = np.mean([standardised_rmse(pw2[:, n], gw2[:, n], fc['std_wl_2d']) for n in range(min(fc['n_2d'], 300))])
        if np.isfinite(r1) and np.isfinite(r2):
            s1.append(r1)
            s2.append(r2)
            n_used += 1
        else:
            n_skipped += 1

        del ev
        gc.collect()

    model.train()
    if len(s1) == 0:
        return float('nan'), float('nan'), float('nan'), n_used, n_skipped
    s1d, s2d = float(np.mean(s1)), float(np.mean(s2))
    return s1d, s2d, 0.5 * (s1d + s2d), n_used, n_skipped

def _set_requires_grad(model, freeze_backbone=False):
    backbone_modules = [
        model.temporal_1d, model.temporal_2d,
        model.static_1d_proj, model.static_2d_proj,
        model.merge_1d, model.merge_2d,
        model.hgt_layers, model.norm_1d, model.norm_2d
    ]
    for m in backbone_modules:
        for p in m.parameters():
            p.requires_grad = not freeze_backbone
    for p in model.head_1d.parameters():
        p.requires_grad = True
    for p in model.head_2d.parameters():
        p.requires_grad = True

def _make_optimizer(model, lr_backbone, lr_head, weight_decay=1e-5):
    backbone_params, head_params = [], []
    head_modules = [model.head_1d, model.head_2d]
    head_param_ids = set()
    for hm in head_modules:
        for p in hm.parameters():
            if p.requires_grad:
                head_params.append(p)
                head_param_ids.add(id(p))
    for p in model.parameters():
        if p.requires_grad and id(p) not in head_param_ids:
            backbone_params.append(p)

    param_groups = []
    if len(backbone_params) > 0:
        param_groups.append({'params': backbone_params, 'lr': lr_backbone})
    if len(head_params) > 0:
        param_groups.append({'params': head_params, 'lr': lr_head})

    return Adam(param_groups, weight_decay=weight_decay)

def _load_backbone_from_model1(model, ckpt_path):
    sd = torch.load(ckpt_path, map_location=DEVICE)
    own = model.state_dict()
    skip_prefixes = ('head_1d.', 'head_2d.')
    copied, skipped = 0, 0
    for k, v in sd.items():
        if k.startswith(skip_prefixes):
            skipped += 1
            continue
        if k in own and own[k].shape == v.shape:
            own[k] = v
            copied += 1
        else:
            skipped += 1
    model.load_state_dict(own, strict=False)
    print(f'Loaded transfer weights: copied={copied}, skipped={skipped}')

def train_one_model(model_id, transfer_from_model1=False, model1_ckpt_path=None, ckpt_name=None):
    print(f'\n{"="*60}')
    print(f'Training Physics-Residual Hybrid for Model {model_id}')
    print(f'{"="*60}')

    fc, static, split = load_model_config(model_id)
    edge_dict = make_edge_dict(static)
    model = build_model(fc)

    if transfer_from_model1:
        assert model1_ckpt_path is not None, 'model1_ckpt_path is required for transfer.'
        _load_backbone_from_model1(model, model1_ckpt_path)

    rain_idx_1d = _find_rain_index(fc['dyn_1d_cols'])
    rain_idx_2d = _find_rain_index(fc['dyn_2d_cols'])
    print(f'Rain feature idx -> 1D: {rain_idx_1d}, 2D: {rain_idx_2d}')

    _set_requires_grad(model, freeze_backbone=transfer_from_model1)
    optimizer = _make_optimizer(
        model,
        lr_backbone=CFG['lr_backbone'] if transfer_from_model1 else CFG['lr'],
        lr_head=CFG['lr_head'] if transfer_from_model1 else CFG['lr'],
        weight_decay=1e-5
    )
    scheduler = CosineAnnealingLR(optimizer, T_max=CFG['epochs'], eta_min=1e-5)
    scaler = amp.GradScaler('cuda', enabled=bool(CFG.get('use_amp', True)))

    best_val, best_epoch = float('inf'), -1
    ckpt = OUT_DIR / (ckpt_name if ckpt_name is not None else f'best_model_{model_id}_physres.pt')
    hist = {'train': [], 'val1': [], 'val2': [], 'val': []}

    for epoch in range(1, CFG['epochs'] + 1):
        t0 = time.time()
        model.train()
        p_teacher = max(0.0, 1.0 - (epoch - 1) / CFG['ss_decay_epochs'])

        if transfer_from_model1 and epoch == (CFG['freeze_backbone_epochs'] + 1):
            _set_requires_grad(model, freeze_backbone=False)
            optimizer = _make_optimizer(
                model,
                lr_backbone=CFG['lr_backbone'],
                lr_head=CFG['lr_head'],
                weight_decay=1e-5
            )
            scheduler = CosineAnnealingLR(optimizer, T_max=CFG['epochs'] - epoch + 1, eta_min=1e-5)
            print('Backbone unfrozen; switched to discriminative LR optimizer.')

        if epoch <= 10:
            ks, ws = CFG['multistep_ks'][:1], CFG['multistep_w'][:1]
        elif epoch <= 20:
            ks, ws = CFG['multistep_ks'][:2], CFG['multistep_w'][:2]
        else:
            ks, ws = CFG['multistep_ks'], CFG['multistep_w']

        paths = split['train'].copy()
        random.shuffle(paths)
        n_events = len(paths)
        ep_loss = 0.0
        n_oom = 0
        optimizer.zero_grad()

        for i, p in enumerate(paths):
            ev = load_event(p)
            try:
                with amp.autocast('cuda', enabled=bool(CFG.get('use_amp', True))):
                    loss = rollout_event(
                        model, ev, static, edge_dict, fc,
                        rain_idx_1d=rain_idx_1d, rain_idx_2d=rain_idx_2d,
                        p_teacher=p_teacher, train=True,
                        multistep_ks=ks, multistep_w=ws
                    )
            except RuntimeError as e:
                if 'out of memory' in str(e).lower():
                    n_oom += 1
                    optimizer.zero_grad(set_to_none=True)
                    torch.cuda.empty_cache()
                    del ev
                    gc.collect()
                    continue
                raise

            if torch.isfinite(loss):
                scaler.scale(loss / CFG['accum_steps']).backward()
                ep_loss += float(loss)
                if (i + 1) % CFG['accum_steps'] == 0 or (i + 1) == n_events:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
            else:
                optimizer.zero_grad(set_to_none=True)

            del ev
            if (i + 1) % 8 == 0:
                torch.cuda.empty_cache()
            gc.collect()

        scheduler.step()
        avg_loss = ep_loss / max(n_events, 1)
        hist['train'].append(avg_loss)

        val_str = ''
        if epoch % 3 == 0 or epoch == CFG['epochs']:
            v1, v2, v, n_used, n_skipped = validate(model, split['val'], static, edge_dict, fc, rain_idx_1d, rain_idx_2d)
            hist['val1'].append(v1)
            hist['val2'].append(v2)
            hist['val'].append(v)
            if np.isfinite(v) and v < best_val:
                best_val, best_epoch = v, epoch
                torch.save(model.state_dict(), ckpt)
                val_str = f' val={v:.4f} (1D={v1:.4f}, 2D={v2:.4f}) [used={n_used}, skipped={n_skipped}] ★ saved'
            else:
                val_str = f' val={v:.4f} (1D={v1:.4f}, 2D={v2:.4f}) [used={n_used}, skipped={n_skipped}]'

        oom_str = f' oom_skips={n_oom}' if n_oom > 0 else ''
        print(f'Epoch {epoch:3d}/{CFG["epochs"]} loss={avg_loss:.5f} p_tf={p_teacher:.2f}{oom_str} {time.time()-t0:.0f}s{val_str}')

    print(f'Best val: {best_val:.4f} at epoch {best_epoch}')
    print(f'Checkpoint: {ckpt}')
    return best_val, ckpt

print('Training loop ready with transfer strategy and validation diagnostics.')

## Cell 6 — Train both models

In [ ]:
results = {}

# Recovery mode: reuse existing Model 1 checkpoint and retrain Model 2 variants.
ckpt_1 = OUT_DIR / 'best_model_1_physres.pt'
assert ckpt_1.exists(), f'Missing Model 1 checkpoint: {ckpt_1}'
print(f'Using existing Model 1 checkpoint: {ckpt_1}')

RUN_MODEL2_TRANSFER = True
RUN_MODEL2_SCRATCH = True  # Set False if you only want transfer-based Model 2.

if RUN_MODEL2_TRANSFER:
    best_val_2_t, ckpt_2_t = train_one_model(
        model_id=2,
        transfer_from_model1=True,
        model1_ckpt_path=ckpt_1,
        ckpt_name='best_model_2_physres_transfer.pt'
    )
    results['model2_transfer'] = {'best_val': best_val_2_t, 'ckpt': str(ckpt_2_t)}

if RUN_MODEL2_SCRATCH:
    best_val_2_s, ckpt_2_s = train_one_model(
        model_id=2,
        transfer_from_model1=False,
        ckpt_name='best_model_2_physres_scratch.pt'
    )
    results['model2_scratch'] = {'best_val': best_val_2_s, 'ckpt': str(ckpt_2_s)}

print('\n=== Model 2 retraining complete ===')
for name, r in results.items():
    print(f'{name}: best val RMSE={r["best_val"]:.4f} | ckpt={r["ckpt"]}')

## Cell 7 — Test inference + submission

In [ ]:
@torch.no_grad()
def test_inference(model_id, ckpt_path):
    fc, static, split = load_model_config(model_id)
    edge_dict = make_edge_dict(static)
    model = build_model(fc)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()

    WARMUP = CFG['warmup_steps']
    L = CFG['lag_window']

    d1_mean, d1_std = fc['depth_mean_1d'], fc['std_depth_1d'] + 1e-8
    d2_mean, d2_std = fc['depth_mean_2d'], fc['std_depth_2d'] + 1e-8
    dd1_std, dd2_std = fc['std_delta_1d'] + 1e-8, fc['std_delta_2d'] + 1e-8

    rain_idx_1d = _find_rain_index(fc['dyn_1d_cols'])
    rain_idx_2d = _find_rain_index(fc['dyn_2d_cols'])

    invert = static['invert_elevation'].cpu().numpy()
    minelev = static['min_elevation'].cpu().numpy()

    out = {}
    for p in tqdm(split['test'], desc=f'M{model_id} inference'):
        ev_id = int(p.stem.split('_')[1])
        ev = load_event(p)
        T = ev['T']
        x1_all = torch.nan_to_num(ev['x_1d_full'].to(DEVICE, non_blocking=True), nan=0.0, posinf=0.0, neginf=0.0)
        x2_all = torch.nan_to_num(ev['x_2d_full'].to(DEVICE, non_blocking=True), nan=0.0, posinf=0.0, neginf=0.0)
        x1 = x1_all.clone()
        x2 = x2_all.clone()
        d1_w = torch.nan_to_num(ev['depth_1d_warmup'].to(DEVICE, non_blocking=True), nan=0.0, posinf=0.0, neginf=0.0).clamp(min=0.0)
        d2_w = torch.nan_to_num(ev['depth_2d_warmup'].to(DEVICE, non_blocking=True), nan=0.0, posinf=0.0, neginf=0.0).clamp(min=0.0)
        time_enc = torch.nan_to_num(ev['time_enc'].to(DEVICE, non_blocking=True), nan=0.0, posinf=0.0, neginf=0.0)

        # Warmup bias calibration
        warm_preds1, warm_preds2 = [], []
        cur_d1, cur_d2 = d1_w[0].clone(), d2_w[0].clone()
        for t in range(WARMUP):
            prev_d1_norm = x1[max(t - 1, 0), :, DEPTH_IDX]
            prev_d2_norm = x2[max(t - 1, 0), :, DEPTH_IDX]
            base1, base2 = physics_baseline_delta_norm(prev_d1_norm, prev_d2_norm, x1[t], x2[t], static, rain_idx_1d, rain_idx_2d)
            x1_win = _make_windows(x1, max(0, t - 1), L)
            x2_win = _make_windows(x2, max(0, t - 1), L)
            d1_tot, d2_tot, _, _ = model.forward_step(
                x1_win, x2_win,
                static['static_1d_node'], static['static_2d_node'],
                time_enc[t], edge_dict, base1, base2
            )
            nd1 = (cur_d1 + d1_tot * dd1_std).clamp(min=0.0)
            nd2 = (cur_d2 + d2_tot * dd2_std).clamp(min=0.0)
            warm_preds1.append(nd1)
            warm_preds2.append(nd2)
            cur_d1, cur_d2 = nd1, nd2

        bias1 = (d1_w - torch.stack(warm_preds1)).mean(0)
        bias2 = (d2_w - torch.stack(warm_preds2)).mean(0)

        pred_d1 = d1_w[-1].clone()
        pred_d2 = d2_w[-1].clone()
        preds1, preds2 = [], []

        for t in range(WARMUP, T):
            prev_d1_norm = x1[t - 1, :, DEPTH_IDX]
            prev_d2_norm = x2[t - 1, :, DEPTH_IDX]
            base1, base2 = physics_baseline_delta_norm(prev_d1_norm, prev_d2_norm, x1[t], x2[t], static, rain_idx_1d, rain_idx_2d)
            x1_win = _make_windows(x1, t - 1, L)
            x2_win = _make_windows(x2, t - 1, L)
            d1_tot, d2_tot, _, _ = model.forward_step(
                x1_win, x2_win,
                static['static_1d_node'], static['static_2d_node'],
                time_enc[t], edge_dict, base1, base2
            )

            nd1 = (pred_d1 + d1_tot * dd1_std + bias1).clamp(min=0.0)
            nd2 = (pred_d2 + d2_tot * dd2_std + bias2).clamp(min=0.0)
            preds1.append(nd1.cpu().numpy())
            preds2.append(nd2.cpu().numpy())

            nd1_norm = (nd1 - d1_mean) / d1_std
            nd2_norm = (nd2 - d2_mean) / d2_std
            t_next = min(t + 1, T - 1)
            x1[t_next] = update_dyn(x1[t], nd1_norm, x1_all[t_next])
            x2[t_next] = update_dyn(x2[t], nd2_norm, x2_all[t_next])
            pred_d1, pred_d2 = nd1, nd2

        out[ev_id] = (np.stack(preds1) + invert[np.newaxis, :], np.stack(preds2) + minelev[np.newaxis, :])
        del ev
        gc.collect()

    return out, fc

def _load_node_ids(model_id, fc):
    p = PREP_ROOT / f'model_{model_id}' / 'node_ids.json'
    if p.exists():
        with open(p) as f:
            d = json.load(f)
        n1, n2 = d.get('node_ids_1d', []), d.get('node_ids_2d', [])
        if len(n1) == fc['n_1d'] and len(n2) == fc['n_2d']:
            return n1, n2
    return list(range(fc['n_1d'])), list(range(fc['n_2d']))

def build_submission(model_ids=(1, 2)):
    rows = []
    row_id = 0

    for mid in model_ids:
        ckpt = OUT_DIR / f'best_model_{mid}_physres.pt'
        if not ckpt.exists():
            print(f'Skipping Model {mid}: missing {ckpt}')
            continue

        preds, fc = test_inference(mid, ckpt)
        node_ids_1d, node_ids_2d = _load_node_ids(mid, fc)

        for ev_id, (p1, p2) in preds.items():
            for t in range(p1.shape[0]):
                for ni, nid in enumerate(node_ids_1d):
                    rows.append({'row_id': row_id, 'model_id': mid, 'event_id': ev_id, 'node_type': 1, 'node_id': nid, 'water_level': float(p1[t, ni])})
                    row_id += 1
                for ni, nid in enumerate(node_ids_2d):
                    rows.append({'row_id': row_id, 'model_id': mid, 'event_id': ev_id, 'node_type': 2, 'node_id': nid, 'water_level': float(p2[t, ni])})
                    row_id += 1

    sub = pd.DataFrame(rows)
    out_path = OUT_DIR / 'submission_physres.csv'
    sub.to_csv(out_path, index=False)
    print(f'Saved: {out_path} | rows={len(sub):,}')
    return sub

# Submission execution is moved to the next cell for flexible checkpoint selection.

## Cell 8 — Inference runs for retrained Model 2 variants

In [ ]:
def build_submission_single_model(model_id, ckpt_path, out_name):
    ckpt_path = Path(ckpt_path)
    assert ckpt_path.exists(), f'Missing checkpoint: {ckpt_path}'

    preds, fc = test_inference(model_id, ckpt_path)
    node_ids_1d, node_ids_2d = _load_node_ids(model_id, fc)

    rows = []
    row_id = 0
    for ev_id, (p1, p2) in preds.items():
        for t in range(p1.shape[0]):
            for ni, nid in enumerate(node_ids_1d):
                rows.append({'row_id': row_id, 'model_id': model_id, 'event_id': ev_id, 'node_type': 1, 'node_id': nid, 'water_level': float(p1[t, ni])})
                row_id += 1
            for ni, nid in enumerate(node_ids_2d):
                rows.append({'row_id': row_id, 'model_id': model_id, 'event_id': ev_id, 'node_type': 2, 'node_id': nid, 'water_level': float(p2[t, ni])})
                row_id += 1

    sub = pd.DataFrame(rows)
    out_path = OUT_DIR / out_name
    sub.to_csv(out_path, index=False)
    print(f'Saved: {out_path} | rows={len(sub):,}')
    return sub

subs = {}
ckpt_m2_transfer = OUT_DIR / 'best_model_2_physres_transfer.pt'
ckpt_m2_scratch = OUT_DIR / 'best_model_2_physres_scratch.pt'

if ckpt_m2_transfer.exists():
    subs['transfer'] = build_submission_single_model(2, ckpt_m2_transfer, 'submission_physres_model2_transfer.csv')

if ckpt_m2_scratch.exists():
    subs['scratch'] = build_submission_single_model(2, ckpt_m2_scratch, 'submission_physres_model2_scratch.csv')

print('Done. Generated submissions:', list(subs.keys()))
for k, df in subs.items():
    print(k, df.head(2))